In [ ]:
# 라이브러리 Import
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)
import xgboost as xgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("="*60)

In [ ]:
product_type_2 = pd.read_csv('../../data_processed/product_type_2.csv', header=[0, 1])

print("="*60)
print("product_type_2 데이터 로드 완료!")
print("="*60)

In [ ]:
# 결측치 확인
print("\n" + "="*60)
print("결측치 확인")
print("="*60)

missing_df = pd.DataFrame({
    '결측수': product_type_2.isnull().sum(),
    '결측비율(%)': (product_type_2.isnull().sum() / len(product_type_2) * 100).round(2)
})
missing_df = missing_df[missing_df['결측수'] > 0].sort_values('결측수', ascending=False)

if len(missing_df) > 0:
    print("\n[결측치 현황]")
    display(missing_df)
else:
    print("\n결측치 없음")

In [ ]:
# 결측치 중앙값 대체
na_cols = product_type_2.columns[product_type_2.isnull().any()]
product_type_2[na_cols] = product_type_2[na_cols].fillna(product_type_2[na_cols].median())

print("="*60)
print("중앙값 대체 완료!!")
print("="*60)
print(f"결측치 데이터 수: {product_type_2.isnull().any().sum()}개")
print("="*60)

In [ ]:
process_data = product_type_2['process']
sensor_data = product_type_2['sensor']

# X값 준비 - process, sensor 컬럼명 평탄화
X = pd.concat([process_data, sensor_data], axis=1)
X.columns = [f"process_{col}" for col in process_data.columns] + \
            [f"sensor_{col}" for col in sensor_data.columns]

# 누수 방지 컬럼 제거
leak_cols = ["process_shot", "defect_flag_is_defect"]
X = X.drop(columns=leak_cols, errors='ignore')
X = X.drop(columns=["sensor_air_pressure_min", "sensor_air_pressure_max", "sensor_coolant_temp_min", "sensor_coolant_temp_max", 
                    "sensor_factory_temp_min", "sensor_factory_temp_max", "sensor_factory_humidity_min", "sensor_factory_humidity_max"], errors="ignore")

# y 값 준비
defects_data = product_type_2['defects']

# 컬럼 범주화
defect_groups = {
    "표면": [
        "stain_1", "stain_2",
        "dent_1", "dent_2",
        "scratch_1", "scratch_2",
        "buring_mark_1", "buring_mark_2",
        "contamination_1", "contamination_2",
        "exfoliation_1", "exfoliation_2",
    ],
    "구조": [
        "short_shot_1", "short_shot_2",
        "bubble_1", "bubble_2",
        "blow_hole_1", "blow_hole_2",
        "deformation_1", "deformation_2",
        "crack_1", "crack_2",
        "impurity_1", "impurity_2",
        "inclusions_1", "inclusions_2",
    ],
}

# 불량유형 범주화 (그룹 내 하나라도 불량이면 1)
y = pd.DataFrame(index=defects_data.index)
for group, cols in defect_groups.items():
    usable = [c for c in cols if c in defects_data.columns]
    y[group] = defects_data[usable].max(axis=1) if usable else 0

print("="*60)
print(y.value_counts().sort_index())
print("="*60)

# 'stratify' 분할 기준용 생성
strata = y.astype(str).agg("".join, axis=1)  # 분할 기준용
print("stratify 분할 기준 데이터:")
print("="*60)
print(strata.value_counts().sort_index())

In [ ]:
X.head()

In [ ]:
# train, test 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=strata
)

print("="*60)
print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

print("="*60)
print("\n[Train set 타겟 분포]")
print("="*60)
print(y_train.value_counts())

print("="*60)
print("\n[Test set 타겟 분포]")
print("="*60)
print(y_test.value_counts())

In [ ]:
plt.subplots(figsize=(30,30))
sns.heatmap(data=X_train.corr(),linewidth=0.1,annot=True,fmt='.2f',cmap='coolwarm')

강한 양의 상관관계
- process_casting_pressure, process_cylinder_pressure (0.90)
    - Shot을 찍을 때의 압력이 Shot을 찍기 위한 실린더 압력으로 거의 그대로 전달되고 있다
- process_pressure_rise_time, process_high_velocity (0.71)
    - 금형에 용탕을 투입하는 속도(high_velocity)가 빠를수록 실린더 압력의 상승 시간도 함께 변동
    - 속도가 빨라지면 시스템 내 부하가 빠르게 걸린다는 물리적 특성
- porcess_spray_2_time, press_high_velocity(0.83)
    - 금형에 용탕을 투입하는 속도(high_velocity)가 빠를수록 스프레이 작업 시간(spray_2_time)이 길어지는 경향
- sensor_factory_temp, process_high_velocity(0.88)
    - 공장 온도 측정값이 높아지면 금형에 용탕을 투입하는 속도(high_velocity)가 빨라진다.
- sensor_factory_temp, process_spray_2_time(0.71)
    - 공장 온도 측정값이 높아지면 스프레이 작업 시간(spray_2_time)이 길어지는 경향
    - 온도가 높으면 금형이 식는 속도가 더디기 때문에 스프레이 시간을 늘려 냉각 보조
sensor_factory_humidity, process_spray_time(0.70)
    - 공장 습도 측정값이 높을수록 스프레이 작업 시간(spray_2_time)이 길어지는 경향
    - 습한 환경에서 냉각 효율을 유지하기 위해 스프레이 시간 조절

강한 음의 상관계수

- process_clamping_force, process_high_velocity (-0.67)
    - Shot을 찍을 때 금형을 밀어주는 힘이 강할수록 금형에 용탕을 투입하는 속도(high_velocity)가 느려진다.
    - 강한 체결력이 금형 내부의 공기 배출을 어렵게 하거나 기계적인 저항을 높여 금형에 용탕을 투입하는 속도에 제동을 거는 물리적 현상
- process_spray_time, process_high_velocity(-0.68)
    - 스프레이 작업 시간(spray_2_time)이 짧을수록 금형에 용탕을 투입하는 속도(high_velocity)가 빨라진다.
- sensor_factory_humidity, process_high_velocity(-0.89)
    - 공장 습도 측정값이 높을수록 금형에 용탕을 투입하는 속도(high_velocity)가 느려진다.
    - 습기가 기계 장비의 마찰 계수를 높이거나 용탕의 열 교환 방식을 변화시켜 사출 속도를 저해
- process_spray_2_time, process_spray_time(-0.80)
    - 스프레이 작업 시간(spray_2_time)이 길을수록 스프레이 작업 시간(spray_time)이 짧아진다.
- sensor_factory_temp, process_spray_time(-0.70)
    - 공장 온도 축정값이 높을수록 스프레이 작업 시간(spray_2_time)이 짧아진다.
- sensor_factory_humidity, process_spray_2_time(-0.77)
    - 공장 습도 측정값이 높을수록 스프레이 작업 시간(spray_2_time)이 짧아진다.
    - 습도가 높으면 공지 중의 수분이 냉각을 보조할 수 있음
- sensor_factory_humidity, sensor_factory_temp(-0.93)
    - 공장 온도 측정값이 높을수록 공장 습도 측정값이 낮아진다.

In [ ]:
# 1. 시각화를 위해 X_train과 y_train을 하나의 데이터프레임으로 합칩니다.
# y_train의 컬럼명을 'target'이라고 가정합니다.
train_df = pd.concat([X_train, y_train], axis=1)

# 2. 피어슨 상관계수 행렬 계산
corr_matrix = train_df.corr()

# 3. 히트맵 시각화
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap: X_train & y_train')
plt.show()

In [ ]:
plt.subplots(figsize=(30,30))
sns.heatmap(data=product_type_2.corr(),linewidth=0.1,annot=True,fmt='.2f',cmap='coolwarm')

In [ ]:
!pip install statsmodels

In [ ]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

# VIF 계산을 위한 함수 정의
def check_vif(X):
    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns
    
    # 각 변수별로 VIF 계산
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) 
                        for i in range(len(X.columns))]
    
    return vif_data.sort_values("VIF", ascending=False)

# 사용 예시 (X_train이 DataFrame 형태여야 함)
vif_result = check_vif(X_train)
display(vif_result)